# Goal-Based Agent for Warehouse Navigation
**Course:** Artificial Intelligence – Agents Laboratory Exercise

This notebook builds a **goal-based agent** that finds a collision-free path for a warehouse vehicle from the start `S` to the goal `G`. The code was generated with the help of an LLM (Claude) and then tested and validated by me.

**Contents**
1. Task 1 – Understanding the problem
2. Task 2 – Designing the agent
3. Task 3 – Prompt engineering (prompt used + implementation)
4. Testing and validation
5. Think About It – scaling up

## Task 1 – Understanding the Problem

**1. What is the environment?**
A 7 × 21 grid-world warehouse. Each cell is either free space (`.`) or an obstacle (`#`, shelving units and outer walls). It is *fully observable* (the whole map is known), *deterministic* (a move always does what it says), *static* (nothing moves while the agent plans), *discrete* and *single-agent*.

**2. What is the goal of the agent?**
To reach the goal cell `G` from the start cell `S` via a collision-free path (no obstacle cells), ideally using the fewest moves.

**3. What actions are available?**
`Up`, `Down`, `Left`, `Right`. Each moves the vehicle one grid square. An action is only legal if the target cell is inside the grid and is not `#`.

**4. What information must the agent maintain to choose its next action?**
- The **current state** (row, column of the vehicle).
- The **goal state** and a **goal test** (is the current cell `G`?).
- The **map / transition model** (which moves are legal and where they lead).
- The **search bookkeeping**: a frontier of states still to explore, a set of already-visited states (to avoid loops), and a parent link for each state so the path can be reconstructed.

**5. Why is this a goal-based agent and not a simple reflex agent?**
A simple reflex agent maps the *current percept* directly to an action through condition–action rules (e.g. "if wall ahead, turn right"). It has no notion of a destination and can loop forever or get stuck in dead ends. This agent instead has an **explicit goal**, a **model of how its actions change the world**, and it **considers sequences of future actions** (search) before committing to a plan. Its actions are selected because they lead towards `G`, not because of a fixed reaction to the immediate percept.

## Task 2 – Designing the Agent

| Component | Design |
|---|---|
| **Environment** | 7 × 21 warehouse grid with walls/shelves (`#`) and free cells (`.`) |
| **Percept / current state** | Vehicle position `(row, col)`; the map is fully known |
| **Goal** | Position of `G` (goal test: `state == goal`) |
| **Actions** | Up `(-1,0)`, Down `(+1,0)`, Left `(0,-1)`, Right `(0,+1)` |
| **Decision-making component** | Breadth-First Search over the state space, which returns a plan (sequence of actions) |

**Block diagram (goal-based agent architecture)**

```mermaid
flowchart LR
    ENV[("Environment<br/>warehouse grid")] -- percepts: position + map --> STATE["State estimate<br/>(current position)"]
    MODEL["World model<br/>'what do my moves do?'<br/>(legal moves, transitions)"] --> PRED
    STATE --> PRED["Predict outcomes<br/>of possible actions"]
    GOAL["Goal<br/>reach G"] --> DECIDE
    PRED --> DECIDE["Decision making<br/>BFS search for shortest path"]
    DECIDE -- next action: U/D/L/R --> ACT["Actuators<br/>move vehicle"]
    ACT --> ENV
```

## Task 3 – Prompt Engineering

### Prompt given to the LLM
> Write a well-documented Python program implementing a goal-based agent for the warehouse navigation problem shown above. The program should
> - represent the warehouse as a two-dimensional grid;
> - determine a collision-free path from S to G;
> - avoid all obstacles;
> - print either the path found or a suitable message if no path exists;
> - explain the search algorithm that has been chosen and why it is appropriate.
>
> *(The map from the lab sheet was pasted after the prompt.)*

### Implementation
The agent is split into three parts: the **Environment** (the grid and its rules), the **GoalBasedAgent** (goal + search) and a **driver** that runs it.

**Chosen algorithm: Breadth-First Search (BFS).**
Every move costs the same (one grid square), so BFS explores states in order of increasing distance from `S`. The first time it reaches `G` it has found a **shortest** path (it is *complete* and *optimal* for uniform step costs). The grid is small, so its memory cost is not a problem.

In [1]:
from collections import deque

WAREHOUSE_MAP = """
#####################
#S....#............G#
#.##....##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################
"""

In [2]:
class WarehouseEnvironment:
    """The warehouse world: a 2D grid of free cells and obstacles."""

    # action name -> (row change, column change)
    ACTIONS = {"Up": (-1, 0), "Down": (1, 0), "Left": (0, -1), "Right": (0, 1)}

    def __init__(self, map_text):
        # Represent the warehouse as a two-dimensional grid (list of lists)
        self.grid = [list(row) for row in map_text.strip().splitlines()]
        self.rows = len(self.grid)
        self.cols = len(self.grid[0])
        if any(len(r) != self.cols for r in self.grid):
            raise ValueError("Map rows must all have the same length.")
        self.start = self._find("S")
        self.goal = self._find("G")

    def _find(self, symbol):
        for r, row in enumerate(self.grid):
            for c, cell in enumerate(row):
                if cell == symbol:
                    return (r, c)
        raise ValueError(f"Map does not contain '{symbol}'.")

    def is_free(self, pos):
        """True if pos is inside the grid and not an obstacle."""
        r, c = pos
        return 0 <= r < self.rows and 0 <= c < self.cols and self.grid[r][c] != "#"

    def successors(self, pos):
        """Transition model: yield (action, new_position) for every legal move."""
        r, c = pos
        for action, (dr, dc) in self.ACTIONS.items():
            new_pos = (r + dr, c + dc)
            if self.is_free(new_pos):
                yield action, new_pos

    def render(self, path=None):
        """Return the map as text, drawing the path with '*' if given."""
        canvas = [row[:] for row in self.grid]
        for (r, c) in (path or []):
            if canvas[r][c] == ".":
                canvas[r][c] = "*"
        return "\n".join("".join(row) for row in canvas)

In [3]:
class GoalBasedAgent:
    """
    A goal-based agent. It has
      * a goal (the goal position),
      * a model of the world (env.successors, i.e. what each action does),
      * a decision-making component (breadth-first search) that plans a
        sequence of actions leading from the current state to the goal.
    """

    def __init__(self, env):
        self.env = env
        self.state = env.start      # current state
        self.goal = env.goal        # explicit goal

    def goal_test(self, state):
        return state == self.goal

    def plan(self):
        """
        Breadth-First Search.
        Returns (actions, positions, nodes_expanded); actions is None if no path exists.
        """
        frontier = deque([self.state])            # FIFO queue -> explores level by level
        parent = {self.state: None}               # also serves as the 'visited' set
        nodes_expanded = 0

        while frontier:
            current = frontier.popleft()
            nodes_expanded += 1

            if self.goal_test(current):
                return (*self._reconstruct(parent, current), nodes_expanded)

            for action, nxt in self.env.successors(current):
                if nxt not in parent:             # never revisit a state
                    parent[nxt] = (current, action)
                    frontier.append(nxt)

        return None, None, nodes_expanded         # frontier exhausted -> unreachable

    @staticmethod
    def _reconstruct(parent, end):
        """Walk parent links back from the goal to the start."""
        actions, positions = [], [end]
        node = end
        while parent[node] is not None:
            node, action = parent[node]
            actions.append(action)
            positions.append(node)
        actions.reverse()
        positions.reverse()
        return actions, positions

    def act(self, actions):
        """Execute the plan one action at a time (moves the vehicle)."""
        for action in actions:
            dr, dc = self.env.ACTIONS[action]
            self.state = (self.state[0] + dr, self.state[1] + dc)

In [4]:
def run_agent(map_text, verbose=True):
    env = WarehouseEnvironment(map_text)
    agent = GoalBasedAgent(env)
    actions, positions, expanded = agent.plan()

    if actions is None:
        if verbose:
            print(f"No path exists from {env.start} to {env.goal}. ({expanded} states explored)")
        return None

    agent.act(actions)
    assert agent.state == env.goal, "Agent did not end at the goal!"
    if verbose:
        print(f"Start: {env.start}   Goal: {env.goal}")
        print(f"Path found with {len(actions)} moves ({expanded} states expanded).\n")
        print("Actions:", " -> ".join(actions), "\n")
        print("Positions (row, col):", positions, "\n")
        print(env.render(positions))
    return actions

path_actions = run_agent(WAREHOUSE_MAP)

Start: (1, 1)   Goal: (1, 19)
Path found with 20 moves (59 states expanded).

Actions: Right -> Right -> Right -> Down -> Right -> Right -> Right -> Up -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right -> Right 

Positions (row, col): [(1, 1), (1, 2), (1, 3), (1, 4), (2, 4), (2, 5), (2, 6), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (1, 16), (1, 17), (1, 18), (1, 19)] 

#####################
#S***.#************G#
#.##****##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################


### Why BFS is appropriate
- **Uniform step cost** – each move costs 1, so BFS's shortest-in-moves path is also the cheapest path.
- **Complete** – if a path exists BFS will find it; if not, it terminates after exhausting the reachable region and reports that no path exists.
- **Optimal** – the first time the goal is dequeued, it has been reached by a shortest path.
- **Simple and cheap for this size** – the map has only a few dozen free cells, so the frontier and visited set are tiny.

## Testing and Validation
Running the program once is not enough. I independently check that the returned path is valid (starts at `S`, ends at `G`, every step is one square, never on `#`), that it is really shortest, and that the "no path" branch works.

In [5]:
def validate_path(map_text, actions):
    """Independently replay the actions on the map and check every rule."""
    env = WarehouseEnvironment(map_text)
    pos = env.start
    for a in actions:
        dr, dc = env.ACTIONS[a]
        pos = (pos[0] + dr, pos[1] + dc)
        assert env.is_free(pos), f"Collision / out of bounds at {pos}"
    assert pos == env.goal, f"Ended at {pos}, not the goal {env.goal}"
    return True

# Test 1: the path from the lab map is valid
assert validate_path(WAREHOUSE_MAP, path_actions)
print("Test 1 passed: path is collision-free and ends at G.")

# Test 2: optimality - compare against a completely independent shortest-path
# calculation (distance labelling flood fill from the start).
def shortest_distance(map_text):
    env = WarehouseEnvironment(map_text)
    dist = {env.start: 0}
    queue = deque([env.start])
    while queue:
        p = queue.popleft()
        for _, n in env.successors(p):
            if n not in dist:
                dist[n] = dist[p] + 1
                queue.append(n)
    return dist.get(env.goal)

assert len(path_actions) == shortest_distance(WAREHOUSE_MAP)
print(f"Test 2 passed: path length {len(path_actions)} equals the independently computed shortest distance.")

# Test 3: unreachable goal -> suitable message (goal walled off)
BLOCKED_MAP = """
#######
#S..#G#
#...###
#######
"""
print("\nTest 3 (blocked goal):")
assert run_agent(BLOCKED_MAP) is None

# Test 4: start next to goal, trivial case
print("\nTest 4 (adjacent):")
assert run_agent("#####\n#SG.#\n#####") == ["Right"]

# Test 5: start == goal cannot be expressed with S and G on the same cell, so
# test that a malformed map (missing G) is rejected with a clear error.
try:
    WarehouseEnvironment("###\n#S#\n###")
except ValueError as e:
    print("\nTest 5 passed: malformed map rejected ->", e)

Test 1 passed: path is collision-free and ends at G.
Test 2 passed: path length 20 equals the independently computed shortest distance.

Test 3 (blocked goal):
No path exists from (1, 1) to (1, 5). (6 states explored)

Test 4 (adjacent):
Start: (1, 1)   Goal: (1, 2)
Path found with 1 moves (2 states expanded).

Actions: Right 

Positions (row, col): [(1, 1), (1, 2)] 

#####
#SG.#
#####

Test 5 passed: malformed map rejected -> Map does not contain 'G'.


## Think About It – What if the warehouse were twice as large?

If both dimensions double, the number of cells grows about **4×**. BFS runs in time and memory proportional to the number of reachable cells, so it would still **work correctly**, and for a few hundred cells it would still be instant. The cell below measures this on synthetic warehouses of increasing size (scale 1, 2, 4, 8).

However, BFS is *uninformed*: it expands in all directions equally, so the frontier becomes a large ring around `S` that grows with the distance to the goal. For much larger maps, a better choice is **A\*** with the Manhattan-distance heuristic, which is admissible for 4-direction grid movement and steers the search towards `G`. On maps where the goal lies in a clear direction it typically expands far fewer states than BFS while still returning an optimal path (the benefit depends on the layout).

**Additional difficulties in a real, larger warehouse:**
- Memory for the frontier/visited set grows; huge maps may need A\*, hierarchical planning or jump-point search.
- The map may be only *partially observable* (sensor range), so the agent must re-plan as it discovers obstacles.
- The environment may be *dynamic* (other vehicles, people, moved pallets), needing replanning or D\* Lite.
- Different move costs (narrow aisles, one-way lanes, turning penalties, battery use) make plain BFS non-optimal, since it treats all moves as cost 1. Uniform-cost search or A\* would be needed.
- Multiple vehicles need coordination to avoid deadlock and collisions.

In [6]:
import time

def make_open_warehouse(scale):
    """Build a larger warehouse by repeating the aisle pattern (scale x wider and taller)."""
    rows, cols = 5 * scale, 20 * scale
    g = [["#"] + ["."] * (cols) + ["#"] for _ in range(rows)]
    for r in range(1, rows, 2):                       # shelving rows with gaps
        for c in range(2, cols - 2):
            if c % 7 != 0:
                g[r][c] = "#"
    g.insert(0, ["#"] * (cols + 2)); g.append(["#"] * (cols + 2))
    g[1][1] = "S"; g[-2][-2] = "G"
    g[1][2] = "."; g[-2][-3] = "."
    return "\n".join("".join(r) for r in g)

print(f"{'scale':>5} {'cells':>7} {'path len':>9} {'states expanded':>16} {'time (ms)':>10}")
for scale in (1, 2, 4, 8):
    m = make_open_warehouse(scale)
    env = WarehouseEnvironment(m)
    t = time.perf_counter()
    acts, _, exp = GoalBasedAgent(env).plan()
    ms = (time.perf_counter() - t) * 1000
    assert acts is not None
    print(f"{scale:>5} {env.rows*env.cols:>7} {len(acts):>9} {exp:>16} {ms:>10.2f}")

scale   cells  path len  states expanded  time (ms)
    1     154        23               72       0.26
    2     504        48              245       0.74
    4    1804        98              950       1.33
    8    6804       198             3720       7.10
